# Entrega 3 · Ampliación: estilos, victorias y ganadores

**Grupo 5K10-05 · Ciencia de Datos · UTN FRM · noviembre 2026**

Completa las preguntas que el clustering descriptivo no contestaba. No sustituye
ni borra `Entrega3_ATP_Estilos_de_juego.ipynb`.

1. ¿Las participaciones de los estilos tienen la misma tasa de victoria?
2. ¿El estilo de ambos jugadores mejora la predicción por encima del ranking?

Una asociación no prueba superioridad causal. Un resultado negativo también responde
la pregunta. Este notebook ejecuta el mismo módulo que genera los parámetros de la app.

## 1. Protocolo fijado antes de evaluar

Entrenar con 2000–2018, elegir C con 2019–2022 y evaluar una vez en 2023–2025.
Las dos alternativas son regresiones logísticas regularizadas comparables: ranking
+ contexto frente a ranking + contexto + estilos. C ∈ {0,01; 0,1; 1; 10}, menor
log-loss en validación. Tras elegir, reajustar hasta 2022. No elegir usando test.

Los centros de carrera completa de E3 incluyen años futuros: **no se usan aquí**.
Se reajusta K-means con perfiles acumulados sólo hasta 2018 y se congela. El criterio
k=3 proviene del estudio descriptivo previo; esta ampliación es retrospectiva y no
un experimento prerregistrado ni una validación externa completamente independiente.

El CSV informa fecha de inicio de torneo. Cada historial excluye todos los partidos
con esa misma fecha, sin intentar inferir el orden diario. No usa estadísticas del
encuentro ni el rol winner/loser como entrada. A/B se asigna al azar, semilla 42.
Requisitos para ambos: ≥20 partidos previos por superficie, coberturas ≥80 %, siete
tasas definidas y rankings válidos. Se excluyen walkovers; retirados conservan su desenlace.

In [1]:
from pathlib import Path
import json, sys
import pandas as pd
import numpy as np
import altair as alt
from IPython.display import display, Markdown
from completar_prediccion import main

informe = main()  # regenera historia, ajustes, selección y evaluación
OUT = Path('resultados/prediccion')
modelo = json.loads((OUT/'modelo_prediccion.json').read_text(encoding='utf-8'))
pd.Series(informe['counts'], name='Cantidad')

{
  "counts": {
    "raw": 77474,
    "valid_surface_non_walkover": 74918,
    "eligible": 40268,
    "train": 29781,
    "validation": 5714,
    "test": 4773,
    "test_candidates": 8423,
    "target_balance_test": 0.4957050073329143
  },
  "metrics": {
    "Ranking + contexto": {
      "accuracy": 0.6446679237376912,
      "auc": 0.6975175602576452,
      "log_loss": 0.6296717102106762,
      "brier": 0.2200732714458487
    },
    "Ranking + contexto + estilos": {
      "accuracy": 0.6417347580138277,
      "auc": 0.6961667171791488,
      "log_loss": 0.6306290020676707,
      "brier": 0.22049778633702555
    },
    "Gana el mejor rankeado": {
      "accuracy": 0.6446679237376912,
      "auc": null,
      "log_loss": null,
      "brier": null
    },
    "Azar 50/50": {
      "accuracy": 0.4957050073329143,
      "auc": 0.5,
      "log_loss": 0.6931471805599453,
      "brier": 0.25
    }
  },
  "comparison": {
    "delta_log_loss": 0.0009572918569944235,
    "log_loss_ci95": [
      -

raw                           77474.000000
valid_surface_non_walkover    74918.000000
eligible                      40268.000000
train                         29781.000000
validation                     5714.000000
test                           4773.000000
test_candidates                8423.000000
target_balance_test               0.495705
Name: Cantidad, dtype: float64

## 2. Historial, grupos y modelos

El script `completar_prediccion.py` es parte de esta entrega, no una caja negra:
`participations` valida los bloques; `history` suma por fecha y resta toda la fecha
actual; `rates` calcula las siete tasas; `fit_styles` ajusta medias/desvíos por
superficie y K-means sobre un perfil por jugador–superficie; `features` arma el contraste.

La base usa `log(rank_B/rank_A)` y sus interacciones con polvo de ladrillo, césped
y best_of=5. El ampliado suma un contraste con signo para cada pareja distinta de
estilos, separado por superficie. No hay intercepto: invertir A/B cambia el signo
de las entradas y hace que P(B)=1−P(A). Emparejamientos iguales aportan cero al
contraste de estilos. Regularización L2 limita los coeficientes.

El escalador y los centros están ajustados hasta 2018, incluso después del reajuste
logístico hasta 2022. No es necesario que las etiquetas coincidan con las de carreras
completas del mapa. Las predicciones de test se hacen en secuencia retrospectiva:
un partido ya disputado en 2023 puede formar el historial de uno de 2024.

In [2]:
display(pd.DataFrame(informe['candidates']))
display(pd.DataFrame(modelo['styles']['centers'], columns=modelo['styles']['columns'], index=modelo['styles']['names']))
display(Markdown('**Seleccionado por validación:** '+informe['selected_by_validation']))

,modelo,C,log_loss_validacion,log_loss_media,log_loss_desvio,accuracy_media,accuracy_desvio,anios
0,Ranking + contexto,0.01,0.632286,0.631612,0.011357,0.628544,0.022105,4
1,Ranking + contexto,0.10,0.632355,0.631685,0.011422,0.628544,0.022105,4
2,Ranking + contexto,1.00,0.632363,0.631694,0.011428,0.628544,0.022105,4
3,Ranking + contexto,10.00,0.632364,0.631695,0.011428,0.628544,0.022105,4
4,Ranking + contexto + estilos,0.01,0.629996,0.629090,0.012361,0.628186,0.020024,4
5,Ranking + contexto + estilos,0.10,0.629910,0.628980,0.012570,0.627979,0.018429,4
6,Ranking + contexto + estilos,1.00,0.629878,0.628944,0.012590,0.627861,0.018717,4
7,Ranking + contexto + estilos,10.00,0.629877,0.628943,0.012591,0.627861,0.018717,4


,tasa_aces,tasa_dobles_faltas,primer_saque_dentro,efectividad_segundo,bp_salvados,puntos_ganados_resto,bp_convertidos
Restadores sólidos,-0.386206,-0.606535,0.438856,0.442275,0.093183,0.564579,0.476330
Saque frágil,-0.329582,0.555402,-0.529819,-0.856295,-0.748424,0.054750,0.050156
Sacadores de potencia,1.092496,0.269326,-0.022377,0.411979,0.852489,-1.021403,-0.867089


**Seleccionado por validación:** Ranking + contexto + estilos

## 3. ¿Hay estilos con más victorias?

Sólo test; una fila es una participación. Cada partido aporta una victoria y una
derrota. Se muestran también resultados por superficie. Los intervalos remuestrean
torneos completos 1.000 veces, evitando tratar todas las participaciones como independientes.
No corrigen todas las dependencias entre años de un mismo jugador. No se equiparan
ranking, rivales o nivel: esto es descriptivo, no una estimación causal.

In [3]:
asociacion = pd.DataFrame(informe['association'])
display(asociacion)
sys.path.insert(0, str(Path('../entrega4').resolve()))
from build import predictive_charts
graficos = predictive_charts(informe)
alt.Chart.from_dict(graficos['victorias']).properties(width=700)

,surface,estilo,n,winrate,lower,upper
0,Todas,Restadores sólidos,6293,0.514222,0.507269,0.520599
1,Todas,Sacadores de potencia,2547,0.486062,0.470985,0.501467
2,Todas,Saque frágil,706,0.423513,0.389259,0.458171
3,cemento,Restadores sólidos,4217,0.517904,0.509949,0.525049
4,cemento,Sacadores de potencia,1740,0.481034,0.464388,0.498117
5,cemento,Saque frágil,441,0.403628,0.361908,0.442667
6,clay,Restadores sólidos,1866,0.504287,0.492124,0.515633
7,clay,Sacadores de potencia,706,0.498584,0.467853,0.532729
8,clay,Saque frágil,192,0.463542,0.387747,0.528851
9,grass,Restadores sólidos,210,0.528571,0.468236,0.561267


alt.LayerChart(...)

### El ranking del partido como control

Los estilos no llegan parejos a los partidos: los restadores son el mejor rankeado
mucho más seguido que saque frágil. Por eso el gráfico abre la tasa de victoria según
quién llega mejor rankeado. La distancia entre filas (más de 20 puntos) supera la que
hay entre estilos dentro de una fila (menos de 10), y entre los peor rankeados el orden
se invierte, con intervalos solapados.

**Este corte se agregó al armar la app, después de ver el test.** Es descriptivo: no
interviene en la selección de C ni del modelo y no cambia ningún coeficiente. Controla
una sola variable y de forma gruesa; no tiene en cuenta la distancia de ranking ni el rival.

In [4]:
display(pd.DataFrame(informe['association_rank']).round(4))
display(pd.Series(informe['best_ranked_share'], name='Llega como mejor rankeado').round(4))

,condicion,estilo,n,winrate,lower,upper
0,Todos sus partidos,Restadores sólidos,6293,0.5142,0.5073,0.5206
1,Todos sus partidos,Sacadores de potencia,2547,0.4861,0.4710,0.5015
2,Todos sus partidos,Saque frágil,706,0.4235,0.3893,0.4582
3,Llega como mejor rankeado,Restadores sólidos,3388,0.6600,0.6411,0.6786
4,Llega como mejor rankeado,Sacadores de potencia,1149,0.6162,0.5875,0.6414
5,Llega como mejor rankeado,Saque frágil,236,0.5636,0.5020,0.6293
6,Llega como peor rankeado,Restadores sólidos,2905,0.3442,0.3285,0.3619
7,Llega como peor rankeado,Sacadores de potencia,1398,0.3791,0.3526,0.4048
8,Llega como peor rankeado,Saque frágil,470,0.3532,0.3094,0.3947


Restadores sólidos       0.5384
Sacadores de potencia    0.4511
Saque frágil             0.3343
Name: Llega como mejor rankeado, dtype: float64

## 4. ¿Aporta el estilo sobre el ranking?

Se comparan exactamente los mismos partidos en ambas alternativas. El piso
determinista «gana el mejor rankeado» tiene accuracy, pero no probabilidades
calibradas para calcular log-loss. Se incluye además P=0,5 como piso probabilístico.
En ese piso, el desempate de accuracy elige A, por eso no da exactamente 50 %.

In [5]:
display(pd.DataFrame(informe['metrics']).T)
display(pd.Series(informe['comparison']))
alt.Chart.from_dict(graficos['rendimiento']).properties(width=700)

,accuracy,auc,log_loss,brier
Ranking + contexto,0.644668,0.697518,0.629672,0.220073
Ranking + contexto + estilos,0.641735,0.696167,0.630629,0.220498
Gana el mejor rankeado,0.644668,NaN,NaN,NaN
Azar 50/50,0.495705,0.500000,0.693147,0.250000


delta_log_loss                                          0.000957
log_loss_ci95     [-0.0004136504830723851, 0.002367090711435759]
delta_accuracy                                         -0.002933
accuracy_ci95        [-0.007303686554486972, 0.0011387252331724]
resamples                                                   2000
tournaments                                                  290
dtype: object

alt.LayerChart(...)

In [6]:
alt.Chart.from_dict(graficos['calibracion']).properties(width=650)

alt.LayerChart(...)

### Dónde cambia algo sumar estilos

Dos modelos sólo difieren en accuracy en los partidos donde eligen favoritos distintos.
Contar esos partidos muestra el tamaño real de la diferencia: el saldo entre lo que
ganan y pierden los estilos en esos casos es exactamente la Δ accuracy.

In [7]:
cambios = informe['favorite_changes']
n = informe['counts']['test']
display(pd.Series(cambios))
assert abs(cambios['net_matches']/n - informe['comparison']['delta_accuracy']) < 1e-12
display(Markdown(f"Sumar estilos cambia al favorito en **{cambios['changed']} de {n}** partidos ({cambios['share']:.2%}). Ahí acierta el ranking {cambios['ranking_right']} veces y los estilos {cambios['styles_right']}: saldo de {cambios['net_matches']} partidos."))

changed          112.000000
share              0.023465
ranking_right     63.000000
styles_right      49.000000
net_matches      -14.000000
dtype: float64

Sumar estilos cambia al favorito en **112 de 4773** partidos (2.35%). Ahí acierta el ranking 63 veces y los estilos 49: saldo de -14 partidos.

### Dónde se equivoca y en qué se apoya

Los errores se miran por grupos de partidos. Para ver en qué se apoya, se apaga un bloque
de coeficientes por vez en el mismo modelo ampliado y se mide la log-loss en el test.

In [8]:
display(pd.DataFrame(informe['mistakes']).round(4))
display(pd.Series(informe['reliance'], name='log-loss en test').round(4))
display(pd.DataFrame(informe['coefficients']).round(3))
alt.Chart.from_dict(graficos['coeficientes']).properties(width=650)

,grupo,categoria,n,accuracy
0,Resultado,Ganó el mejor rankeado,3077,0.9795
1,Resultado,Ganó el peor rankeado,1696,0.0289
2,Distancia de ranking,Rankings parecidos,1591,0.5449
3,Distancia de ranking,Distancia media,1593,0.6177
4,Distancia de ranking,Distancia grande,1589,0.7627
5,Superficie,Cemento,3199,0.6436
6,Superficie,Polvo de ladrillo,1382,0.6382
7,Superficie,Césped,192,0.6354
8,Formato,Tres sets,4030,0.6280
9,Formato,Cinco sets,743,0.7160


completo       0.6306
sin_estilos    0.6295
sin_ranking    0.6922
Name: log-loss en test, dtype: float64

,variable,coef,bloque
0,Ventaja de ranking,0.498,Ranking
1,Ranking × polvo de ladrillo,0.004,Ranking
2,Ranking × césped,-0.048,Ranking
3,Ranking × cinco sets,0.237,Ranking
4,Cemento: Restadores vs. Saque frágil,0.236,Estilos
5,Cemento: Restadores vs. Sacadores,-0.007,Estilos
6,Cemento: Saque frágil vs. Sacadores,-0.215,Estilos
7,Polvo de ladrillo: Restadores vs. Saque frágil,0.294,Estilos
8,Polvo de ladrillo: Restadores vs. Sacadores,0.194,Estilos
9,Polvo de ladrillo: Saque frágil vs. Sacadores,0.010,Estilos


alt.Chart(...)

## 5. Conclusión y límites

La calibración muestra probabilidades medias y frecuencias reales en ocho grupos
cuantiles; no se ajusta un calibrador con el test. Los intervalos de diferencias
son pareados por torneo (2.000 repeticiones, 290 torneos).

El estilo es una descripción útil. En este protocolo, agregarlo no muestra mejora
clara por encima del ranking. El intervalo incluye cero: no demuestra igualdad,
superioridad causal ni que los estilos jamás puedan aportar. El ampliado ganó en
validación y se conserva como modelo elegido, aunque no mejore en test.

Limitaciones: cobertura restringida a jugadores con historial; cambios de época;
fecha de torneo conservadora; clasificación de estilos débil; k heredado de la
exploración retrospectiva; dependencia entre jugadores; sólo regresión logística
y esta familia de interacciones evaluadas. No prometer acertar un encuentro particular.

In [9]:
a=informe['metrics']['Ranking + contexto']['accuracy']
b=informe['metrics']['Ranking + contexto + estilos']['accuracy']
ci=informe['comparison']['accuracy_ci95']
display(Markdown(f"**Resultado:** ranking + contexto {a:.2%}; + estilos {b:.2%}. Diferencia {(b-a)*100:+.3f} puntos porcentuales; IC95 % [{ci[0]*100:+.3f}; {ci[1]*100:+.3f}]."))
test = pd.read_csv(OUT/'predicciones_test.csv')
display(test[['fecha','nombre_a','nombre_b','estilo_a','estilo_b','p_ranking','p_estilos','gana_a']].tail(10))

**Resultado:** ranking + contexto 64.47%; + estilos 64.17%. Diferencia -0.293 puntos porcentuales; IC95 % [-0.730; +0.114].

,fecha,nombre_a,nombre_b,estilo_a,estilo_b,p_ranking,p_estilos,gana_a
4763,2025-11-19,Matteo Berrettini,Jurij Rodionov,Sacadores de potencia,Restadores sólidos,0.643226,0.641072,1
4764,2025-11-20,Jan-Lennard Struff,Tomas Martin Etcheverry,Sacadores de potencia,Restadores sólidos,0.457024,0.459806,0
4765,2025-11-20,Francisco Cerundolo,Alexander Zverev,Restadores sólidos,Restadores sólidos,0.269606,0.274941,0
4766,2025-11-20,Pablo Carreno Busta,Jakub Mensik,Restadores sólidos,Sacadores de potencia,0.311980,0.315170,0
4767,2025-11-20,Jaume Munar,Jiri Lehecka,Restadores sólidos,Sacadores de potencia,0.405096,0.406020,1
4768,2025-11-21,Flavio Cobolli,Zizou Bergs,Restadores sólidos,Restadores sólidos,0.584975,0.582722,1
4769,2025-11-22,Pablo Carreno Busta,Jan-Lennard Struff,Restadores sólidos,Sacadores de potencia,0.492597,0.491161,1
4770,2025-11-22,Alexander Zverev,Jaume Munar,Restadores sólidos,Restadores sólidos,0.781200,0.775266,1
4771,2025-11-23,Pablo Carreno Busta,Matteo Berrettini,Restadores sólidos,Sacadores de potencia,0.440958,0.440924,0
4772,2025-11-23,Jaume Munar,Flavio Cobolli,Restadores sólidos,Restadores sólidos,0.437275,0.438953,0


## 6. Lo que llega a la Entrega 4

`modelo_prediccion.json` contiene escalador, centros y coeficientes; `informe.json`
contiene el protocolo, las métricas y los intervalos; `predicciones_test.csv` lleva
casos históricos con las tasas previas. `test_temporal.py` verifica exclusión actual/
futura; `entrega4/test_app.cjs` compara las probabilidades Python y JavaScript.
Los datos crudos se identifican por SHA-256. La app nunca reentrena al mover un filtro.